# L12a: Model-Free Reinforcement Learning and Q-Learning
In this lecture, we explore the fundamentals of model-free reinforcement learning, a branch of machine learning focused on how an agent should take actions in an environment to maximize its cumulative reward over time when it has no model of that environment.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the reinforcement learning problem:__ An agent observes the state of its environment, takes an action, and receives a reward and a new state, without knowing the rules that produced them. We'll contrast this setting with value iteration, which needs those rules written down as transition probabilities and rewards.
> * __Update action values with temporal differences:__ Q-learning stores the value of every action in every state in a table, and after each step it corrects one entry by the difference between what it observed and what it had predicted. We'll build this update from the Bellman equation and assemble it into the Q-learning algorithm.
> * __Explain when Q-learning converges:__ Q-learning finds the optimal action values if it keeps trying every action in every state and its learning rate shrinks at the right pace. We'll see how epsilon-greedy exploration and the learning rate meet these conditions, and what goes wrong in practice when they do not.

At its core, reinforcement learning is about learning from doing: an agent observes the state of its environment, takes actions, i.e., makes decisions, receives rewards, and updates its knowledge to improve its future decisions. Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Learn a path through the lava grid world with Q-learning](CHEME-5800-L12a-Example-QLearningGridWorld-Fall-2026.ipynb). In this example, a robot learns to reach a charging station and avoid lava pits from its own moves, without being told where either one is.
>
> We train the robot with Q-learning, then compare its policy with the optimal policy that value iteration computes from the known rewards and moves.

___

## Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s\in\mathcal{S}$ and can take an action $a\in\mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$ and moves to a new state $s^{\prime}$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-Schematic-RL.svg" width="580" style="max-width:100%; height:auto;" alt="An agent box above an environment box. An arrow labeled action a runs from the agent to the environment. Two arrows run back from the environment to the agent, one labeled reward r and one labeled state s to s prime."/>
</p>

__Agent and environment.__ The agent observes its current state $s$ and selects an action $a$. The environment answers with a reward $r$ and the next state $s^{\prime}$. The agent uses what it observed to improve its next decision.

Previously, we explored two approaches to learning from interaction:
* __Multiplicative weights__ adjusts the probability of selecting each action based on its past performance, in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight, even in a changing environment. That is, it minimizes regret.
* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.

__Q-learning__ adds the state. It estimates the long-term value (utility) of each state-action pair, which lets the agent learn good behavior when its actions change what happens next. All three approaches must balance the same fundamental challenge: exploring new actions to gather information versus exploiting known actions to collect reward.

___

## From Value Iteration to Action Values
Before we develop Q-learning, let's review the value functions of a Markov decision process (MDP), the tuple $\left(\mathcal{S}, \mathcal{A}, R(s,a), T\left(s^{\prime}\,|\,s,a\right), \gamma\right)$ of states, actions, rewards, transition probabilities, and a discount factor, and see exactly where value iteration uses its model of the world.

Value iteration computes the __optimal value function__ $U^{*}(s)$ (utility, satisfaction, etc.) by repeatedly applying the __Bellman backup__:

$$
\begin{equation*}
U_{k+1}(s) = \max_{a\in\mathcal{A}}\left(\underbrace{R(s,a)}_{\text{= now}} +
\gamma\;\overbrace{\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})}^{\text{= future}}\right)
\end{equation*}
$$

where $k$ counts the __sweeps__, complete passes of the backup over every state, and $U_{k}(s)\to U^{*}(s)$ as $k\to\infty$ for $0\leq\gamma<1$. The quantity inside the maximum scores one action. It is the __action-value function__ (also called the Q-function), given by:

$$
\begin{equation*}
Q^{*}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U^{*}}(s^{\prime})
\end{equation*}
$$

where $Q^{*}(s,a)$ is the expected cumulative discounted reward of taking action $a$ in state $s$ and following the optimal policy afterward. The best action value in a state is that state's value, $U^{*}(s)=\max_{a\in\mathcal{A}}Q^{*}(s,a)$, and the optimal policy picks the action that achieves it, $\pi^{*}(s)=\arg\max_{a\in\mathcal{A}}Q^{*}(s,a)$. Substituting the first relation into the definition of $Q^{*}(s,a)$ gives the __Bellman equation__ for the action values:

$$
\begin{equation*}
Q^{*}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot\max_{a^{\prime}\in\mathcal{A}}Q^{*}(s^{\prime},a^{\prime})
\end{equation*}
$$

Each of these equations needs the reward function $R(s,a)$ and the transition model $T\left(s^{\prime}\,|\,s,a\right)$. When we have them, value iteration requires $O(|\mathcal{S}|^{2}\cdot|\mathcal{A}|)$ operations per sweep. When we do not, which is the usual case for a real robot, a reactor, or a patient, we cannot evaluate the sum over the next states at all.

We store the action values in a table, with one row per state and one column per action:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-Q-Schematic.svg" width="580" style="max-width:100%; height:auto;" alt="A grid of dots with states as rows and actions as columns. The dot in row state i and column action j is highlighted as utility (i,j). In the row for state s, the dot in column action k is highlighted as the maximum utility for state s."/>
</p>

__The Q-table.__ Entry $(i,j)$ holds the value (utility) of taking action $j$ in state $i$. The policy reads each row and picks the column with the largest entry. For state $s$, that is action $k$.

___

## Q-Learning
Q-learning, introduced by [Watkins (1989)](https://www.cs.rhul.ac.uk/~chrisw/new_thesis.pdf), learns the action-value function directly from experience, without $R(s,a)$ or $T\left(s^{\prime}\,|\,s,a\right)$. It estimates $Q(s,a)$ by conducting repeated experiments in the world.

In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment, the agent updates its estimate of $Q(s,a)$ using the update rule:

$$
\begin{equation*}
Q(s,a)\gets\underbrace{Q(s,a)}_{\text{old value}}+\alpha\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime}) - Q(s,a)\right)}_{\text{new information}}
\end{equation*}
$$

where $0<\alpha\leq{1}$ is the learning rate, and $0\leq\gamma<{1}$ is the discount factor. Only the entry for the pair $(s,a)$ that was tried changes. Every other entry keeps its value.

__Why does this update make sense?__ Compare the term $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$ with the right side of the Bellman equation. The agent cannot compute the sum over next states. Instead, it uses the one next state $s^{\prime}$ that the world produced and the one reward $r$ that it received.

This one-sample estimate of the right side is called the __temporal difference (TD) target__. The bracketed term of the update is the __temporal difference error__:

$$
\begin{equation*}
\delta = r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime}) - Q(s,a)
\end{equation*}
$$

where $\delta$ is the gap between the value the agent predicted before the step and the value it can estimate one step later.

The update moves the old value a fraction $\alpha$ of the way toward the target. Two cases show what this step does:
* __A single outcome:__ If each action has a single outcome and $\alpha=1$, the update replaces $Q(s,a)$ with the target. This is a Bellman backup of one state-action pair, with the observed next state in place of the sum.
* __A random outcome:__ If the outcome is random, the target changes from visit to visit. The update then averages the targets. With a learning rate that shrinks as the visits add up, such as $\alpha=1/n$ on the $n$-th visit, this average approaches the expected value that the sum over $T\left(s^{\prime}\,|\,s,a\right)$ would have computed.

We estimate the policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action that maximizes $Q(s,a)$ in each state $s$:

$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

### Algorithm
Q-learning runs in __episodes__. Each episode starts the agent in some state and lets it act until it enters an absorbing state, such as the charging station or a lava pit of a grid world, or until a step limit.

Each step chooses an action with the epsilon-greedy rule: with probability $\epsilon$ the agent explores with a random action, and otherwise it exploits the action with the largest current value.

__Initialize__: Given the state space $\mathcal{S}$, the action space $\mathcal{A}$, the absorbing states, a discount factor $0\leq\gamma<1$, a learning rate $0<\alpha\leq 1$, an exploration probability $0\leq\epsilon\leq 1$, a number of episodes $N\geq 1$, and a maximum number of steps per episode $d\geq 1$, set $Q(s,a)\gets 0$ for all $s\in\mathcal{S}$ and $a\in\mathcal{A}$.

For $i = 1$ to $N$ __do__:
1. Choose a starting state $s$ that is not absorbing, for example uniformly at random, and set the step counter $t\gets 0$.
2. While $s$ is not absorbing and $t<d$ __do__:
    - Choose the action $a$: roll a random number $p\in[0,1]$.
        - If $p\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime})$ (exploitation).
    - Take action $a$, and observe the reward $r$ and the next state $s^{\prime}$ from the world.
    - Update the action value with the temporal difference error:
      $$Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$$
    - Update the state $s\gets s^{\prime}$ and the step counter $t\gets t+1$.

__Extract Policy__: For each state $s\in\mathcal{S}$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a)$.

An episode ends as soon as the agent enters an absorbing state. The agent never updates the rows of $Q$ that belong to absorbing states. Those rows stay at zero, and the target of a move into an absorbing state is just its reward $r$, with no future to add.

### Convergence
Q-learning converges to the optimal action-value function, $Q(s,a)\to Q^{*}(s,a)$ for every pair, with probability one under the following conditions ([Watkins and Dayan, 1992](https://doi.org/10.1007/BF00992698)), assuming the environment has the Markov property:
* __Finite, fixed MDP and discounting__: The state and action spaces are finite. This lets $Q$ fit in a table. The rewards are bounded, the rewards and the transition probabilities do not change over time, and the discount factor satisfies $0\leq\gamma<1$.
* __Learning rate decay__: Let $\alpha_{n}(s,a)$ denote the learning rate used for the $n$-th update of the pair $(s,a)$. For every pair, $\sum_{n=1}^{\infty}\alpha_{n}(s,a)=\infty$ and $\sum_{n=1}^{\infty}\alpha_{n}^{2}(s,a)<\infty$.

  The first sum lets the updates carry $Q(s,a)$ as far from its starting value as it needs to go. The second sum makes the noise of the individual targets average out.

  The choice $\alpha_{n}(s,a)=1/n$ satisfies both. With this choice, $Q(s,a)$ is the running average of the targets the pair has seen.
* __Infinite exploration__: Every state-action pair of a state that is not absorbing is visited, and updated, infinitely often. The rows of absorbing states stay at zero.

  Epsilon-greedy action selection with an $\epsilon$ that stays above zero, together with episodes that can start in any state that is not absorbing, meets this condition, because every episode has a chance to start in $s$ and try $a$ first.

A constant learning rate violates the second condition. Its effect depends on whether the targets are random:
* __Random rewards or next states:__ The estimates keep responding to each new target. They fluctuate around $Q^{*}(s,a)$ instead of settling on it. A smaller $\alpha$ gives smaller fluctuations.
* __A single outcome per action:__ This is the case in this week's examples and labs. The targets carry no noise. A constant learning rate then also settles on $Q^{*}(s,a)$.

Either way, the greedy policy is already correct once the best action in each state has the largest estimate.

### Caveats & Practical Notes
The infinite-exploration condition is the difficult one in practice, and three issues come up again and again:
* __Exploration schedule__: If the exploration probability $\epsilon$ is too small, or falls too fast, the agent stops trying actions before it has learned their values, and it can settle on a poor action for good. If $\epsilon$ stays large, the agent keeps wasting steps on random actions. This is the exploration versus exploitation trade-off we met with bandits.
* __Sparse and delayed rewards__: When few state-action pairs earn a nonzero reward, an agent that knows nothing acts at random and may wander for a long time before it sees one. The reward then reaches earlier states only one update at a time, carried backward by the $\max$ term. Reward shaping, which adds small rewards that point toward the goal, is one remedy.
* __Rate of convergence__: Convergence is guaranteed, but its speed is not. Because every state-action pair needs many visits, the number of experiments grows with the size of the table. A problem with a known model is usually solved faster by value iteration.

___

## Lab
In the [lab](../L12b/CHEME-5800-L12b-Lab-AdaptiveDosing-QLearning-Fall-2026.ipynb), we write our own Q-learning and use it to learn a chemotherapy dosing policy. We check the learned doses against the optimal policy computed by value iteration.

___

## Summary
In this lecture, we developed model-free reinforcement learning, in which an agent learns how to act from the rewards it receives:

> __Key Takeaways:__
>
> * __Model-free learning needs only experience:__ We described reinforcement learning as an agent that observes states, takes actions, and receives rewards, without the transition probabilities and rewards that value iteration needs. Learning from the steps it actually takes lets the agent act in a world that no one has written down as a model.
>
> * __Q-learning corrects one action value per step:__ We wrote the Bellman equation for the action values and replaced its sum over next states with the one next state the agent observed. Each update moves the stored value toward this sampled target by a fraction set by the learning rate, and the greedy policy reads the best action from each row of the table.
>
> * __Convergence needs exploration and a shrinking learning rate:__ Q-learning is guaranteed to find the optimal action values when every action is tried again and again in every state that is not absorbing, and the learning rate shrinks slowly enough to travel far but quickly enough to average out the noise. Epsilon-greedy action selection and a learning rate that falls with the visit count meet these conditions.

Reinforcement learning lets an agent learn good behavior through trial and error, even when no one can write down the rules of its environment.

___